# 05 · Promo events: uplift and post-promo dip (PySpark + numpy)

An **event** is a run of consecutive promotion days for one store-item. **Uplift** = actual units ÷ (baseline ×
days) − 1. **Post-promo dip** is the same over the 7 days after (voided if another promotion overlaps).
**Net lift** combines both: promotion + the week after vs. the baseline for both. Did it pay back?
Each family gets a **median with a bootstrap 95% CI**: resample the events 1,000 times, take the middle 95%.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath("../src"))  # the repo's src/, via the Databricks Git folder

from favorita_spark.config import Config

cfg = Config()
spark.sql(f"USE CATALOG {cfg.catalog}")

In [ ]:
from favorita_spark.promo import family_summary, promo_events

t = spark.table
promo_events(t("gold.fact_sales"), t("gold.dim_date"), t("gold.stg_item"), t("gold.stg_store_day"), cfg) \
    .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.fact_promo_event")

In [ ]:
events = spark.table("gold.fact_promo_event").select("family", "uplift", "post_dip", "net_lift").toPandas()
summary = family_summary(events, cfg)
spark.createDataFrame(summary).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.promo_family_summary")
display(summary)